# Allocation and contribution analysis

This example uses a model of global petrochemical production: 155 processes and
114 objects, running from feedstocks through primary chemicals and polymers to
fertiliser use and end-of-life treatment. It is built here at one scenario's
parameter values, and then used to ask two kinds of question:

- how many GHG emissions are allocated to each product?
- and where does that burden come from?

In [ ]:
import pandas as pd

%run load_model.py

len(model.processes), len(model.objects)

## Allocating burdens

Assume we want to allocate burdens between all co-products by mass, apart from air, water and wastes
which are allocated nothing.

The model tracks several types of elementary exchanges: the greenhouse gases emitted directly by processes (`CO2`, `CH4`, `N2O`), the cradle-to-gate burdens of feedstocks, electricity and process heat entering the system, and a `CO2_captured` diagnostic. We use `characterise={"GWP": GWP}` to add one more column combining them into kg CO2e.

In [ ]:
from flowprog.allocation import AllocatedSystem, ByValue, Excluding, ProcessSet

RESIDUAL_OUTPUTS = {
    "Air",
    "MiscRefineryProducts",
    "Nutrients",
    "PyrolysisResidue",
    "Waste",
    "WasteBiomass",
    "WasteOtherChemicals",
    "Water",
}

allocated = AllocatedSystem(
    model,
    params,
    rule=Excluding(RESIDUAL_OUTPUTS, ByValue()),
    characterise={"GWP": GWP},
)
allocated

The resulting allocated system includes emissions intensities for all objects:

In [ ]:
POLYMERS = [
    "HDPEPolyethylene",
    "LDPEPolyethylene",
    "PPPolypropylene",
    "PSPolystyrene",
    "PVCPolyvinylChloride",
    "PETPolyethyleneTerephthalatePolyesters",
]

allocated.object_intensities.loc[
    POLYMERS, ["GWP", "CO2", "GHG_upstream_Feedstock", "GHG_upstream_ProcessHeat"]
].round(1)

## Where does a burden come from?

PVC is a good example: it is made both from virgin vinyl chloride and by recycling, so there is something to compare.

A contribution analysis breaks one burden down by choosing a set of processes to *expand*. Those are reported with their own direct emissions, while any inputs from elsewhere are collapsed into a cradle-to-gate burden of that input.

The smallest useful choice of processes to expand is just the processes that make PVC itself:

In [ ]:
PVC = "PVCPolyvinylChloride"

first_tier = ProcessSet.producers_of(structure, PVC)
sorted(first_tier)

In [ ]:
breakdown = allocated.contributions(object=PVC, expand=first_tier)
breakdown.upstream[["GWP", "CO2", "GHG_upstream_ProcessHeat"]].round()

With this allocation choice, the EOL PVC being recycled carries no burden, but some electricity is required to recycle it.

Most of the burdens however come from primary production, of which most is associated with the production of vinyl chloride.

In this case, neither process has any direct emissions:

In [ ]:
breakdown.direct[["GWP"]]

The breakdown always adds up to the original total burden:

In [ ]:
breakdown.check()
assert breakdown.total["GWP"] == allocated.object_intensities.loc[PVC, "GWP"]
breakdown.total["GWP"]

## Expanding one step further

Most of that burden arrived as vinyl chloride. Adding its producer to the set
opens that step up: vinyl chloride disappears as a line of its own, replaced by
the emissions of the process that makes it and by what *that* process took in.

In [ ]:
expanded = first_tier | ProcessSet.producers_of(structure, "VinylChloride")

deeper = allocated.contributions(object=PVC, expand=expanded)
deeper.upstream[["GWP"]].round()

Two of those inputs come out as zero. They are reported as `unsupplied`: the processes that could make chlorine and oxygen are not running in this scenario,
so they appear in the system with no burden.

In [ ]:
deeper.unsupplied()

## Reporting in categories

Individual inputs are often too fine-grained to report. `with_group` collects
them into named categories, checking on the way that the groups are disjoint
and that nothing has been forgotten. Direct emissions have no input object, so
they become a category of their own:

In [ ]:
grouped = deeper.with_group(
    "category",
    {
        "chlorine": ["Chlorine"],
        "other chemicals": ["Ethylene", "InorganicAcids", "PureOxygen"],
        "recycled feedstock": ["PVCPolyvinylChlorideAtEOL"],
        "electricity": ["Electricity", "LowCarbonElectricity"],
        "process heat": ["ProcessHeat"],
    },
    direct_label="direct process emissions",
)
grouped.by("category")[["GWP"]].round()

## Comparing the two routes

`object_intensities` gives the average across everything producing an object.
Naming a process as well breaks down that object *as made by that route*, which
is what separates recycled PVC from virgin:

In [ ]:
pvc_shares = allocated.supply_shares.query("object == @PVC").set_index("process")["sigma"]
pvc_shares

In [ ]:
pd.DataFrame({
    "GWP": {
        process: allocated.contributions(
            object=PVC, process=process, expand=ProcessSet.of(process)
        ).total["GWP"]
        for process in pvc_shares.index
    },
    "share of supply": pvc_shares,
})

The average is the supply-weighted mix of the two.

## The whole chain, with utilities collapsed

Expanding everything upstream attributes the burden to the processes that
physically emitted it. `upstream_of` builds that set by walking the model
structure, and `stopping_at_objects` leaves parts of it collapsed -- here the
utilities, so that electricity and heat are reported as inputs rather than
followed back to their sources:

In [ ]:
UTILITIES = ["Electricity", "LowCarbonElectricity", "ProcessHeat"]

whole_chain = ProcessSet.upstream_of(
    structure, object=PVC, stopping_at_objects=UTILITIES
)
len(whole_chain)

In [ ]:
chain = allocated.contributions(object=PVC, expand=whole_chain)

by_object = chain.by("object")["GWP"]
by_object[by_object != 0].sort_values(ascending=False).round(1)

`direct` here is now the the emissions of all the expanded processes, summed together – everything *except* supply of process heat and electricity.

By looking at the breakdown by process, we can see that the largest are the refining and cracking steps:

In [ ]:
chain.direct["GWP"].sort_values(ascending=False).head(8).round(1)

Those are the total GWP (kgCO2e) — we can also see the breakdown by different exchange types:

In [ ]:
chain.by("object")[by_object != 0].round(1)

It might make sense to report feedstocks separately:

In [ ]:
feedstock_GHGs = chain.direct["GHG_upstream_Feedstock"]
feedstock_GHGs[feedstock_GHGs > 0]

In [ ]:
whole_chain_up_to_feedstocks = whole_chain - {k for k in whole_chain if k.startswith("OilRefining") or k.startswith("SourceOf")}

chain2 = allocated.contributions(object=PVC, expand=whole_chain_up_to_feedstocks)
chain2

In [ ]:
chain2.by("object")[chain2.by("object")["GWP"] != 0].sort_values("GWP").round(3)

Now each feedstock object is clearly reported with the GHG_upstream_Feedstock emissions type, while the CO2 and CH4 emissions are all in the residual "direct" group as shown above.